# GPS OD-Flow Training

Train GPS models with different configurations. Weights and metrics are saved to `results/`
and consumed by `benchmark.ipynb` for comparison.

**Architecture options:**
- `encoder_type`: `'gps'` (GPS-GNN) | `'mlp'` (TransFlower)
- `pe_type`: `'rwpe'` | `'spe'` | `'rrwp'` | `'lape'` (WeDAN dual Laplacian)
- `loss_type`: `'huber'` | `'ce'` | `'multitask'` | `'zinb'` | `'focal'`
- `use_rle`: adds Relative Location Encoder


In [ ]:
import sys, os, gc, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

sys.path.insert(0, str(Path('.').resolve()))
warnings.filterwarnings('ignore')

from models.GPS.config import (
    TrainingConfig, device, MC_EPOCHS,
    SINGLE_CITY_ID, MULTI_CITY_IDS, RESULTS_DIR, METRICS_CSV, WEIGHTS_DIR,
    ensure_dirs,
)
from models.GPS.data_load import prepare_single_city_data, prepare_multi_city_data
from models.GPS.main import train_single_city, train_multi_city
from models.GPS.metrics import evaluate_full_matrix, compute_metrics

ensure_dirs()
print(f"Device: {device}")
print(f"Results dir: {RESULTS_DIR}")


## GPS Configurations

Edit this cell to select which experiments to run.

In [ ]:
# ── Single-city run configurations ──────────────────────────────────────────
# Format: run_id -> (display_name, TrainingConfig)
SC_CONFIGS = {
    # Core baselines
    'B1':  ('B1: GPS+Bilinear+Huber (raw)',   TrainingConfig(decoder_type='bilinear', loss_type='huber', prediction_mode='raw')),
    'B2':  ('B2: GPS+TF+Huber (raw)',          TrainingConfig(decoder_type='transflower', loss_type='huber', prediction_mode='raw')),
    'B6':  ('B6: GPS+Bilinear+CE (norm)',       TrainingConfig(decoder_type='bilinear', loss_type='ce', prediction_mode='normalized', use_dest_sampling=False)),
    'B7':  ('B7: GPS+TF+CE (norm)',             TrainingConfig(decoder_type='transflower', loss_type='ce', prediction_mode='normalized', use_dest_sampling=False)),
    # PE ablations
    'B7+lape': ('B7+LaPE: GPS+TF+CE+LaPE (norm)', TrainingConfig(decoder_type='transflower', loss_type='ce', prediction_mode='normalized', use_dest_sampling=False, pe_type='lape')),
    'B7+spe':  ('B7+SPE: GPS+TF+CE+SPE (norm)',   TrainingConfig(decoder_type='transflower', loss_type='ce', prediction_mode='normalized', use_dest_sampling=False, pe_type='spe')),
    # Loss ablations
    'B7+focal': ('B7+Focal: GPS+TF+Focal (norm)', TrainingConfig(decoder_type='transflower', loss_type='focal', prediction_mode='normalized', use_dest_sampling=False)),
    'B7+rle':   ('B7+RLE: GPS+TF+CE+RLE (norm)',  TrainingConfig(decoder_type='transflower', loss_type='ce', prediction_mode='normalized', use_dest_sampling=False, use_rle=True)),
    # TransFlower (MLP encoder)
    'TF1':       ('TF1: TransFlower+CE (norm)',         TrainingConfig(encoder_type='mlp', decoder_type='transflower', loss_type='ce', prediction_mode='normalized', use_dest_sampling=False)),
    'TF1+rle':   ('TF1+RLE: TransFlower+CE+RLE (norm)', TrainingConfig(encoder_type='mlp', decoder_type='transflower', loss_type='ce', prediction_mode='normalized', use_dest_sampling=False, use_rle=True)),
    'TF1+focal': ('TF1+Focal: TransFlower+Focal (norm)', TrainingConfig(encoder_type='mlp', decoder_type='transflower', loss_type='focal', prediction_mode='normalized', use_dest_sampling=False)),
}

# ── Multi-city run configurations ────────────────────────────────────────────
MC_CONFIGS = {
    'C1':       ('C1: MC GPS+TF+Huber (raw)',    TrainingConfig(decoder_type='transflower', loss_type='huber', prediction_mode='raw', mc_epochs=MC_EPOCHS)),
    'C2':       ('C2: MC GPS+TF+CE (norm)',       TrainingConfig(decoder_type='transflower', loss_type='ce', prediction_mode='normalized', use_dest_sampling=True, include_zero_pairs=False, mc_epochs=MC_EPOCHS)),
    'C2+lape':  ('C2+LaPE: MC GPS+TF+CE+LaPE',   TrainingConfig(decoder_type='transflower', loss_type='ce', prediction_mode='normalized', use_dest_sampling=True, include_zero_pairs=False, pe_type='lape', mc_epochs=MC_EPOCHS)),
    'C2+focal': ('C2+Focal: MC GPS+TF+Focal',     TrainingConfig(decoder_type='transflower', loss_type='focal', prediction_mode='normalized', use_dest_sampling=False, mc_epochs=MC_EPOCHS)),
    'C2+rle':   ('C2+RLE: MC GPS+TF+CE+RLE',      TrainingConfig(decoder_type='transflower', loss_type='ce', prediction_mode='normalized', use_dest_sampling=True, include_zero_pairs=False, use_rle=True, mc_epochs=MC_EPOCHS)),
    'TC1':      ('TC1: MC TransFlower+CE (norm)',  TrainingConfig(encoder_type='mlp', decoder_type='transflower', loss_type='ce', prediction_mode='normalized', use_dest_sampling=False, mc_epochs=MC_EPOCHS)),
    'TC1+rle':  ('TC1+RLE: MC TransFlower+CE+RLE', TrainingConfig(encoder_type='mlp', decoder_type='transflower', loss_type='ce', prediction_mode='normalized', use_dest_sampling=False, use_rle=True, mc_epochs=MC_EPOCHS)),
}

print(f"Single-city configs: {len(SC_CONFIGS)}")
print(f"Multi-city configs:  {len(MC_CONFIGS)}")


## Single-City Training

In [ ]:
# ── Data loading with pe_type caching ────────────────────────────────────────
_sc_cache = {}

def get_sc_data(pe_type='rwpe'):
    if pe_type not in _sc_cache:
        print(f"  Loading single-city data (pe_type={pe_type})...")
        _sc_cache[pe_type] = prepare_single_city_data(pe_type=pe_type)
        N = _sc_cache[pe_type]['num_nodes']
        tm = _sc_cache[pe_type]['train_mask'].sum()
        print(f"  N={N}, train_pairs={tm}")
    return _sc_cache[pe_type]


In [ ]:
sc_results = {}

for run_id, (run_name, cfg) in SC_CONFIGS.items():
    city_data = get_sc_data(cfg.pe_type)
    try:
        result = train_single_city(run_id, run_name, cfg, city_data=city_data)
        sc_results[run_id] = result
    except Exception as e:
        print(f"  ERROR {run_id}: {e}")
    finally:
        if device.type == 'cuda':
            torch.cuda.empty_cache()
            gc.collect()

print(f"\nCompleted: {len(sc_results)} single-city runs")


## Multi-City Training

In [ ]:
# ── Multi-city data loading with pe_type caching ─────────────────────────────
_mc_cache = {}

def get_mc_data(pe_type='rwpe'):
    if pe_type not in _mc_cache:
        print(f"  Loading multi-city data (pe_type={pe_type})...")
        city_data_dict, train_ids, val_ids, test_ids = prepare_multi_city_data(pe_type=pe_type)
        _mc_cache[pe_type] = (city_data_dict, train_ids, val_ids, test_ids)
        print(f"  Train: {train_ids}  Val: {val_ids}  Test: {test_ids}")
    return _mc_cache[pe_type]


In [ ]:
mc_results = {}

for run_id, (run_name, cfg) in MC_CONFIGS.items():
    city_data_dict, train_ids, val_ids, test_ids = get_mc_data(cfg.pe_type)
    try:
        result = train_multi_city(
            run_id, run_name, cfg,
            city_data_dict=city_data_dict,
            train_city_ids=train_ids,
            val_city_ids=val_ids,
        )
        mc_results[run_id] = result
    except Exception as e:
        print(f"  ERROR {run_id}: {e}")
    finally:
        if device.type == 'cuda':
            torch.cuda.empty_cache()
            gc.collect()

print(f"\nCompleted: {len(mc_results)} multi-city runs")


## Results Summary

In [ ]:
def print_results_table(results, label):
    if not results:
        print(f"  No {label} results.")
        return
    print(f"\n{'='*70}")
    print(f"  {label} Results")
    print(f"{'='*70}")
    print(f"  {'Run ID':<18} {'CPC_full':>9} {'CPC_nz':>9} {'CPC_test':>9} {'MAE':>9}  Status")
    print(f"  {'-'*66}")
    for rid, r in results.items():
        mf = r.get('metrics_full', {})
        mnz = r.get('metrics_nonzero', {})
        mt = r.get('metrics_test_pairs', {})
        status = r.get('status', '?')
        print(f"  {rid:<18} {mf.get('CPC', float('nan')):>9.4f} "
              f"{mnz.get('CPC', float('nan')):>9.4f} "
              f"{mt.get('CPC', float('nan')):>9.4f} "
              f"{mf.get('MAE', float('nan')):>9.4f}  {status}")

print_results_table(sc_results, "Single-City")
print_results_table(mc_results, "Multi-City")

# Show saved metrics CSV
if METRICS_CSV.exists():
    df = pd.read_csv(METRICS_CSV)
    print(f"\n  Total rows in metrics.csv: {len(df)}")
    print(f"  Saved weights: {len(list(WEIGHTS_DIR.glob('*.pt')))}")


In [ ]:
# Training curves
def plot_histories(results, title):
    if not results:
        return
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(title)
    for rid, r in results.items():
        h = r.get('history', {})
        if not h:
            continue
        axes[0].plot(h.get('train_loss', []), label=rid, alpha=0.8)
        axes[1].plot(h.get('val_loss', []), label=rid, alpha=0.8)
        axes[2].plot(h.get('val_cpc_full', []), label=rid, alpha=0.8)
    axes[0].set_title('Train Loss'); axes[0].set_xlabel('Epoch')
    axes[1].set_title('Val Loss');   axes[1].set_xlabel('Epoch')
    axes[2].set_title('CPC (full)'); axes[2].set_xlabel('Epoch')
    for ax in axes:
        ax.legend(fontsize=7, ncol=2)
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_histories(sc_results, "Single-City Training Histories")
plot_histories(mc_results, "Multi-City Training Histories")
